In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import re

df = pd.read_csv("data/house_prices.csv")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (187531, 21)


,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [11]:
df.info()
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 0])

<class 'pandas.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  str    
 2   Description        184508 non-null  str    
 3   Amount(in rupees)  187531 non-null  str    
 4   Price (in rupees)  169866 non-null  float64
 5   location           187531 non-null  str    
 6   Carpet Area        106858 non-null  str    
 7   Status             186916 non-null  str    
 8   Floor              180454 non-null  str    
 9   Transaction        187448 non-null  str    
 10  Furnishing         184634 non-null  str    
 11  facing             117298 non-null  str    
 12  overlooking        106095 non-null  str    
 13  Society            77853 non-null   str    
 14  Bathroom           186703 non-null  str    
 15  Balcony            138596 non-null  str    
 16  Car Parking  

In [ ]:
price_col = [c for c in df.columns if "amount" in c.lower() or "price" in c.lower()][0]

def parse_amount(x):
    if not isinstance(x, str):
        if pd.isna(x):
            return None
        return float(x)
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        elif "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", "").strip())
    except ValueError:
        return None

df["price_clean"] = df[price_col].apply(parse_amount)
df = df.dropna(subset=["price_clean"]).copy()

area_col = [c for c in df.columns if "carpet" in c.lower() or "area" in c.lower()][0]

def parse_area(x):
    if not isinstance(x, str):
        return float(x) if not pd.isna(x) else np.nan
    match = re.search(r"([\d\.]+)\s*(sqft|sqm)", str(x), re.IGNORECASE)
    if match:
        val, unit = float(match.group(1)), match.group(2).lower()
        return val * 10.764 if unit == "sqm" else val
    return np.nan

df["carpet_area_sqft"] = df[area_col].apply(parse_area)
df["carpet_area_sqft"] = df["carpet_area_sqft"].fillna(df["carpet_area_sqft"].median())

floor_col = [c for c in df.columns if "floor" in c.lower()][0] if any("floor" in c.lower() for c in df.columns) else None

def parse_floor(x):
    if not isinstance(x, str):
        return 1
    match = re.search(r"\b(\d+)\b", str(x))
    return int(match.group(1)) if match else 1

df["floor_num"] = df[floor_col].apply(parse_floor) if floor_col else 1

bath_col = [c for c in df.columns if "bath" in c.lower()][0] if any("bath" in c.lower() for c in df.columns) else None
balc_col = [c for c in df.columns if "balcony" in c.lower()][0] if any("balcony" in c.lower() for c in df.columns) else None

if bath_col:
    df["bathroom"] = pd.to_numeric(df[bath_col].astype(str).str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int)
else:
    df["bathroom"] = 1

if balc_col:
    df["balcony"] = pd.to_numeric(df[balc_col].astype(str).str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int)
else:
    df["balcony"] = 1

loc_col = [c for c in df.columns if "loc" in c.lower()][0] if any("loc" in c.lower() for c in df.columns) else df.columns[0]
top_locations = df[loc_col].value_counts().head(50).index.tolist()
df["location_grouped"] = df[loc_col].apply(lambda x: x if x in top_locations else "Other")

furnish_col = [c for c in df.columns if "furnish" in c.lower()]
trans_col = [c for c in df.columns if "transaction" in c.lower()]
owner_col = [c for c in df.columns if "owner" in c.lower()]
facing_col = [c for c in df.columns if "facing" in c.lower()]

df["Furnishing"] = df[furnish_col[0]].fillna("Semi-Furnished") if furnish_col else "Semi-Furnished"
df["Transaction"] = df[trans_col[0]].fillna("Resale") if trans_col else "Resale"
df["Ownership"] = df[owner_col[0]].fillna("Ready to Move") if owner_col else "Ready to Move"
df["facing"] = df[facing_col[0]].fillna("North") if facing_col else "North"

q_low = df["price_clean"].quantile(0.01)
q_high = df["price_clean"].quantile(0.99)
df_clean = df[(df["price_clean"] >= q_low) & (df["price_clean"] <= q_high)].copy()

print("Cleaned Dataset Rows:", len(df_clean))
df_clean.head()

Cleaned Dataset Rows: 175484


,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,Ownership,Super Area,Dimensions,Plot Area,price_clean,carpet_area_sqft,floor_num,bathroom,balcony,location_grouped
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,Ready to Move,NaN,NaN,NaN,4200000.0,500.0,10,1,2,thane
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,Freehold,NaN,NaN,NaN,9800000.0,473.0,3,2,1,thane
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,Freehold,NaN,NaN,NaN,14000000.0,779.0,10,2,1,thane
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,Ready to Move,NaN,NaN,NaN,2500000.0,530.0,1,1,1,thane
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,Co-operative Society,NaN,NaN,NaN,16000000.0,635.0,20,2,1,thane


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

X = df_clean[numeric_features + categorical_features]
y = df_clean["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = Pipeline([("prep", preprocessor), ("reg", LinearRegression())])
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

model = Pipeline([("prep", preprocessor), ("reg", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Linear Regression MAE:", mean_absolute_error(y_test, base_pred))
print("Linear Regression R2:", r2_score(y_test, base_pred))

print("Random Forest MAE:", mean_absolute_error(y_test, pred))
print("Random Forest R2:", r2_score(y_test, pred))

joblib.dump(model, "../models/house_price.pkl")

locations_sorted = sorted(df_clean["location_grouped"].unique().tolist())
with open("../models/locations.json", "w") as f:
    json.dump(locations_sorted, f)

print("Export Complete!")

Linear Regression MAE: 4442303.498271493
Linear Regression R2: 0.5875471483775843
Random Forest MAE: 1100290.4721926467
Random Forest R2: 0.9087792478808826
Export Complete!
